# Exploring the SAP-RPT-Decisions API (`/systemone`)

The Decision API answers decision tasks: you describe a `state` — for example, a ticket, an e-mail, or a statement — and ask one or more questions about it. The model behind the Decision API returns probabilities over user-defined options.

There are three different types of questions:

| type | asks | answer |
|------|------|--------|
| `noul`   | A yes/no judgement | Probability the statement is true (0..1) |
| `choice` | Choose one of the options | Selected option + `probabilities` + `confidence` |
| `score`  | Score on an ordered scale (lowest first) | Probability-weighted mean level + `legend` + `probabilities` + `confidence` |

The model learns in context, which makes it behave like a **few-shot learner** (not zero-shot like other decision models).
**You must pass a list of labeled `examples`** (at least two), and every answer label
you declare must appear in at least one example, since the model can only predict labels it has seen.

## Setup

Built and tested on Python 3.14. Install dependencies before running the notebook (once per environment):

In [ ]:
!pip install -r requirements.txt

In [ ]:
import json  # pretty-printing API responses throughout the notebook
import os  # environment variables and path inspection

## Configuring the API

Set your AI Core deployment URL in the cell below, then add your BTP service key to a `.env` file in the same directory:

```
AICORE_SERVICE_KEY={"clientid": "...", "clientsecret": "...", "url": "https://..."}
```

Make sure `.env` is listed in your `.gitignore` so secrets are not accidentally committed.

Requests are scoped to an AI Core **resource group** (defaults to `default`). If your deployment uses a different one, you can change it here.

In [ ]:

DEPLOYMENT_URL = "<Set your AI Core deployment URL here>"

os.environ["AICORE_RESOURCE_GROUP"] = "default" 

In [ ]:
from dotenv import load_dotenv  # loads secrets from .env into os.environ

load_dotenv()

if not os.environ.get("AICORE_SERVICE_KEY"):
    raise EnvironmentError(
        "AICORE_SERVICE_KEY not set. Create a .env file with:\n"
        "AICORE_SERVICE_KEY={...your service key JSON...}"
    )

In [ ]:
from rpt_decision_api import RptDecisionClient  # SAP-RPT Decision API client

_client = RptDecisionClient(url=DEPLOYMENT_URL.rstrip("/") + "/systemone")
print(f"Routing /systemone calls to: {_client.url}")
print(f"Using AI Core resource group: {os.environ.get('AICORE_RESOURCE_GROUP', 'default')}")

## Example 1: Choosing banking intent

The [Banking77](https://huggingface.co/datasets/PolyAI/banking77) dataset (CC-BY) contains
online banking customer-service queries across 77 fine-grained intent options
([Casanueva et al., 2020](https://arxiv.org/abs/2003.04807)).

We use a subset of 4 intents to showcase the Decision API on this
real classification dataset. Every intent must appear as an answer in at least one
example. Otherwise, the model can't make decisions.

In the banking example, the state is described by a string, for example "My card payment doesn't work.", for which we want the model to answer "What is the customer's banking intent?" with one of four possible options: `card_arrival`, `lost_or_stolen_card`, `pin_blocked`, `declined_card_payment`. The options may be further described, which we did manually here (not part of the original Banking77) in the `criteria` field.

In [ ]:
BANKING_INTENTS = {
    "card_arrival":            "Asking when a newly ordered card will arrive",
    "lost_or_stolen_card":     "Reporting a lost or stolen card",
    "pin_blocked":             "PIN blocked after too many wrong attempts",
    "declined_card_payment":   "Card payment was declined",
}

payload_banking = {
    "model": "sap-rpt-decisions",
    "state": "My card payment doesn't work.",
    "questions": {
        "intent": {
            "type": "choice",
            "instructions": "What is the customer's banking intent?",
            "criteria": BANKING_INTENTS,
        }
    },
    "examples": [
        {"state": "What can I do if my card still hasn't arrived after 2 weeks?", "answers": {"intent": "card_arrival"}},
        {"state": "I ordered my card but it still isn't here", "answers": {"intent": "card_arrival"}},
        {"state": "Can you track my card for me?", "answers": {"intent": "card_arrival"}},

        {"state": "Help Me!  I lost my card.  What should I do?.", "answers": {"intent": "lost_or_stolen_card"}},
        {"state": "Someone stole my wallet with my card in it while vacationing in Spain. I really need my card blocked and a new one ordered.", "answers": {"intent": "lost_or_stolen_card"}},
        {"state": "My card has gone missing.", "answers": {"intent": "lost_or_stolen_card"}},

        {"state": "I have exceeded the number of PIN attempts", "answers": {"intent": "pin_blocked"}},
        {"state": "I tried entering my pin several times incorrectly. Why isn't my pin working?", "answers": {"intent": "pin_blocked"}},
        {"state": "Why was my PIN blocked?", "answers": {"intent": "pin_blocked"}},

        {"state": "My card is not working at stores.", "answers": {"intent": "declined_card_payment"}},
        {"state": "The card payment I attempted to make failed.", "answers": {"intent": "declined_card_payment"}},
        {"state": "I NEED HELP WITH MY CARD!!!!! please. It keeps getting declined when I try to use it. I have tried a few times to use it and it keeps getting rejected. I have no idea why.", "answers": {"intent": "declined_card_payment"}},
    ],
}

result = _client.ask_raw(payload_banking)
print(json.dumps(result, indent=2))

For this question, the model correctly classified the intent as `declined_card_payment` with 90% probability.
Note that we only used 3 examples per option. In our current case, the possible options are the four intents.
In general, more examples are better as it gives the decision model more context to correctly classify the state.

The `answers` object is keyed by question name (in this case just one question). Since our question was of type `choice`, we get:

- **`choice`** — the selected option.
- **`probabilities`** — the full distribution across all options (sums to 1).
- **`confidence`** — how decisive the distribution is: `0` = flat (no signal),
  `1` = all mass on one option.

## Example 2: Multiple questions (`noul` and `score`)

In this fictive example, we want to make decisions based on customer complaints.
Instead of asking just one question as in the example above, we ask two questions:

- The urgency of the complaint which can be low, high, or critical. This is a question of type `score`, since there is an order to the options.
- Whether human intervention is required or not, a question of type `noul`, because it is either true or false.

In [ ]:
URGENCY_LEVELS = [
    "Low",      # 0 — minor inconvenience or purely informational, no active harm
    "High",     # 1 — customer can't use the product or service as intended
    "Critical", # 2 — safety risk, fraud, complete loss, or legal exposure
]

payload = {
    "model": "sap-rpt-decisions",
    "state": "Just wanted to say thank you!",
    "questions": {
        "urgency": {
            "type": "score",
            "instructions": "How urgent is the customer's issue?",
            "criteria": URGENCY_LEVELS,
        },
        "requires_human_reply": {
            "type": "noul",
            "instructions": "Does the customer expect or need a direct human response?",
            "criteria": {
                "true":  "Asks a question, requests an action (refund, replacement, callback), or is waiting on a resolution",
                "false": "Shares feedback, leaves a review, or makes a statement with no expectation of a personal response",
            },
        },
    },
    "examples": [
        # Low urgency, no reply needed — pure feedback / reviews
        {"state": "Delivery was two days late but everything looks good.",
         "answers": {"urgency": 0, "requires_human_reply": False}},
        {"state": "The colour doesn't quite match the photos. A bit disappointing but I'll keep it.",
         "answers": {"urgency": 0, "requires_human_reply": False}},

        # Low urgency, reply needed — minor issue but customer asks for something
        {"state": "Manual was missing from the box. Can you email me a copy?",
         "answers": {"urgency": 0, "requires_human_reply": True}},

        # High urgency, reply needed — blocked and requesting action
        {"state": "Stopped working after a week. I need a replacement or a refund.",
         "answers": {"urgency": 1, "requires_human_reply": True}},
        {"state": "App crashes on every login attempt. I cannot access my account at all.",
         "answers": {"urgency": 1, "requires_human_reply": True}},

        # High urgency, no reply needed — broken but not asking for help
        {"state": "Half the parts were missing. Returned it at the store already.",
         "answers": {"urgency": 1, "requires_human_reply": False}},

        # Critical, reply needed — demanding action
        {"state": "I was charged three times for one order and nobody responds. I need this fixed now.",
         "answers": {"urgency": 2, "requires_human_reply": True}},
        {"state": "Package never arrived and support ghosted me for a month. Where is my refund?",
         "answers": {"urgency": 2, "requires_human_reply": True}},

        # Critical, no reply needed — reporting, not requesting
        {"state": "The device overheated and burned my hand. I've already filed a report with consumer safety.",
         "answers": {"urgency": 2, "requires_human_reply": False}},
    ],
}

result = _client.ask_raw(payload)
print(json.dumps(result, indent=2))

The returned object's `answers` field now contains the results for both questions.

Since "urgency" was a question of type `score`, we get the mean score (field `score`), a legend with the labels (field `legend`), the probability distribution over the possible output scores, and the confidence level, which is scaled between 0 (low confidence) and 1 (high confidence).

The question "requires_human_reply" is of type `noul` and therefore has a much simpler object structure. We only get the probability for whether the answer is yes — in our case, the probability that the customer complaint needs human intervention. No separate confidence level is provided.

## Example 3: Structured `state`

The `state` does not have to be a single string. It can be a **structured object**, and
its fields can be **numeric or categorical** and not just free text. This matters when the
decision depends on values that no amount of prose captures well.

For this example, we take a fictive case of classifying financial transactions:
each transaction is described by its amount, the card's country, the country it was
used in, and how old the account is. We ask a **single `noul` question**: is this
transaction likely fraudulent?

Notice there is no free-text `state` at all. The signal lives entirely in the structured
fields: a large amount, a country mismatch, and a brand-new account together look like
fraud, while the very same merchant on an established, domestic account does not.

In [ ]:
payload = {
    "model": "sap-rpt-decisions",
    "state": {
        "amount_usd": 2100.00,
        "merchant_category": "Electronics",
        "card_country": "US",
        "transaction_country": "BR",
        "account_age_days": 4,
    },
    "questions": {
        "is_fraud": {
            "type": "noul",
            "instructions": "Is this transaction likely fraudulent?",
            "criteria": {
                "true": "Unusual amount, a card/transaction country mismatch, or a very new account",
                "false": "Amount, location and account age are consistent with normal use",
            },
        },
    },
    "examples": [
        # Domestic, modest amount, long-lived account -> legit
        {"state": {"amount_usd": 42.50, "merchant_category": "Groceries",
                   "card_country": "US", "transaction_country": "US", "account_age_days": 1200},
         "answers": {"is_fraud": False}},
        {"state": {"amount_usd": 15.99, "merchant_category": "Streaming",
                   "card_country": "DE", "transaction_country": "DE", "account_age_days": 640},
         "answers": {"is_fraud": False}},
        # Foreign, but established account and a travel-sized amount -> still legit
        {"state": {"amount_usd": 310.00, "merchant_category": "Travel",
                   "card_country": "US", "transaction_country": "FR", "account_age_days": 900},
         "answers": {"is_fraud": False}},

        # Large amount, country mismatch, brand-new account -> fraud
        {"state": {"amount_usd": 1890.00, "merchant_category": "Electronics",
                   "card_country": "US", "transaction_country": "RU", "account_age_days": 30},
         "answers": {"is_fraud": True}},
        {"state": {"amount_usd": 2500.00, "merchant_category": "Jewelry",
                   "card_country": "GB", "transaction_country": "NG", "account_age_days": 7},
         "answers": {"is_fraud": True}},
        # Large amount on a days-old account, even without a country mismatch -> fraud
        {"state": {"amount_usd": 3200.00, "merchant_category": "Electronics",
                   "card_country": "US", "transaction_country": "US", "account_age_days": 2},
         "answers": {"is_fraud": True}},
    ],
}

result = _client.ask_raw(payload)
print(json.dumps(result, indent=2))